# Distilling Expert Knownldge using LLM

Why selecting chess? Chess is very hard, Lichess chess database has trillions of positions & Stockfish is close to an oracle
So, no way you can memorize the moves, but you need to learn strategy, that is not simple.
This project uses the concept of on-policy distillation, to distill the expert knowldge into an LLM model.

The notebook walks through the training math used in this repo — **no training is run**. One fixed chess
position is used throughout, and every concept is shown on a small worked example with the
intermediate values printed, so each number can be checked by hand.

| § | Topic |
|---|-------|
| 0 | Setup: Qwen3-1.7B (CPU, bfloat16), python-chess, Stockfish, the fixed FEN |
| 1 | Where this fits: SFT vs RL/GRPO vs on-policy distillation |
| 2 | GRPO loss on hand-set toy tensors (ratio, clip, KL, aggregation) + hand verification |
| 3 | GRPO on a real sampled group from Qwen3-1.7B, scored by Stockfish |
| 4 | On-policy distillation: per-token reverse KL over a student trace |
| 5 | Side-by-side: how dense is each training signal? |
| 6 | Takeaways |

**Runtime:** a few minutes on a CPU-only laptop. **First run downloads `Qwen/Qwen3-1.7B` (~3.4 GB)**
from the Hugging Face Hub into the local HF cache; later runs are offline-fast. Same default base model as the rest of this repo (`scripts/train_sft.sh`, `SFTConfig.model_name`).

**Environment:** CPU-only torch — see the install line in the repo's `notebooks/README.md`
(`uv venv` + `uv pip install torch --index-url https://download.pytorch.org/whl/cpu transformers ...`).

## Refrences
1. [Rethinking On-Policy Distillation of Large Language
Models II: One Training Example](https://arxiv.org/pdf/2609.04172)
2. [DeepSeekMath: Pushing the Limits of Mathematical Reasoning in Open Language Models](https://arxiv.org/pdf/2402.03300)
3. [Learning to Reason under Off-Policy Guidance](https://arxiv.org/abs/2504.14945)
4. [Learning to Replicate Expert Judgment in Financial Tasks](https://thinkingmachines.ai/news/learning-to-replicate-expert-judgment-in-financial-tasks/)
5. [Why Salesforce may be AI's adult in the room](https://www.thedeepview.com/articles/why-salesforce-may-be-ai-s-adult-in-the-room?utm_source=thedeepview&utm_medium=newsletter&utm_campaign=ai-labs-agree-on-testing-but-enforcement-is-harder)

## 0. Setup

Imports and global constants. Everything is seeded so the notebook is reproducible run-to-run
(CPU sampling with a fixed `torch.manual_seed` is deterministic). Figures are written to
`notebooks/figures/` **and** shown inline.

In [2]:
import math
import os
import re
import statistics
from pathlib import Path

import chess
import chess.engine
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer

SEED = 0
torch.manual_seed(SEED)
torch.set_num_threads(max(1, os.cpu_count() or 1))
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 20)

FIG_DIR = Path("figures")
FIG_DIR.mkdir(exist_ok=True)

# One position for the whole notebook: Italian Game, White to move.
FEN = "r1bqk2r/pppp1ppp/2n2n2/2b1p3/2B1P3/2NP1N2/PPP2PPP/R1BQK2R w KQkq - 4 5"
board = chess.Board(FEN)
print(board)
print()
print("Side to move:", "White" if board.turn else "Black", "| legal moves:", board.legal_moves.count())

r . b q k . . r
p p p p . p p p
. . n . . n . .
. . b . p . . .
. . B . P . . .
. . N P . N . .
P P P . . P P P
R . B Q K . . R

Side to move: White | legal moves: 40


![italian move](figures/italian-game-5-o-o.gif)

### Stockfish

Stockfish is the reward model / judge. The binary lives **outside** the repo at `~/.local/bin/stockfish`.
We use one persistent engine with `Threads=1` and a fixed depth so evaluations are deterministic.

The reward is the repo's actual design (`qwen_chess_rl/training/rl/reward.py`, `docs/rewards.md`):

$$r = Q(s,a) = P(\text{mover wins} \mid \text{position after the move}) \in [0,1], \qquad p = \frac{1}{1 + 10^{-cp/400}}$$

Stockfish scores the position *after* the move from the **opponent's** perspective (they are now to move),
so the mover's reward is `1 − winprob_after`. Mate ⇒ 1.0, illegal / unparseable move ⇒ −1.0 floor.

In [3]:
STOCKFISH_PATH = os.path.expanduser("~/.local/bin/stockfish")
if not (os.path.isfile(STOCKFISH_PATH) and os.access(STOCKFISH_PATH, os.X_OK)):
    raise FileNotFoundError(
        f"Stockfish not found/executable at {STOCKFISH_PATH}. Install Stockfish 17+ "
        "(e.g. download from https://stockfishchess.org/download/) and place the binary there."
    )

EVAL_DEPTH = 12          # repo default (docs/rewards.md)
CP_SCALE = 400.0         # Lichess-scale logistic
ILLEGAL_REWARD = -1.0    # hard floor for illegal / unparseable moves

engine = chess.engine.SimpleEngine.popen_uci(STOCKFISH_PATH)
engine.configure({"Threads": 1, "Hash": 16})
print(engine.id["name"])


def score_to_win_prob(score: chess.engine.Score) -> float:
    # Win probability for the side to move (mirrors reward.py::_score_to_win_prob).
    if score.is_mate():
        return 1.0 if score.mate() > 0 else 0.0
    return 1.0 / (1.0 + 10.0 ** (-score.score() / CP_SCALE))


def move_reward(fen: str, uci: str | None) -> dict:
    # r = 1 - winprob_after (mover's perspective); -1.0 if illegal.
    b = chess.Board(fen)
    try:
        move = chess.Move.from_uci(uci) if uci else None
    except ValueError:
        move = None
    if move is None or move not in b.legal_moves:
        return {"move": uci, "san": "illegal", "legal": False, "cp_after": None, "reward": ILLEGAL_REWARD}
    san = b.san(move)
    b.push(move)
    info = engine.analyse(b, chess.engine.Limit(depth=EVAL_DEPTH))
    score = info["score"].relative          # relative to the opponent (now to move)
    reward = 1.0 - score_to_win_prob(score)
    cp_mover = -score.score(mate_score=100_000)
    return {"move": uci, "san": san, "legal": True, "cp_after": cp_mover, "reward": reward}


root_info = engine.analyse(board, chess.engine.Limit(depth=EVAL_DEPTH))
BEST_UCI = root_info["pv"][0].uci()
BEST_SAN = board.san(root_info["pv"][0])
ROOT_CP = root_info["score"].relative.score(mate_score=100_000)
print(f"Stockfish (depth {EVAL_DEPTH}) best move: {BEST_SAN} ({BEST_UCI}), eval {ROOT_CP:+d} cp for White")

Stockfish 16
Stockfish (depth 12) best move: Nd5 (c3d5), eval +44 cp for White


### Load Qwen3-1.7B (CPU, bfloat16)

The same base model the repo trains by default (`Qwen/Qwen3-1.7B`, see `SFTConfig.model_name` /
`docs/training.md`). Loaded on CPU in bfloat16 — no CUDA, no vLLM. `eval()` mode; we never call
`.backward()` in this notebook.

> **First run note:** this downloads ~3.4 GB of weights into `~/.cache/huggingface`.

In [4]:
MODEL_ID = "Qwen/Qwen3-1.7B"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.bfloat16)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"{MODEL_ID}: {n_params:,} parameters ({n_params / 1e6:.0f}M), dtype={next(model.parameters()).dtype}")
_eos = model.generation_config.eos_token_id
EOS_IDS = set(_eos if isinstance(_eos, list) else [_eos])
print("EOS ids:", sorted(EOS_IDS), "| pad id:", tokenizer.pad_token_id)

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Qwen/Qwen3-1.7B: 1,720,574,976 parameters (1721M), dtype=torch.bfloat16
EOS ids: [151643, 151645] | pad id: 151643


## 1. Where this fits: SFT vs RL (GRPO) vs on-policy distillation

Three ways to turn expert judgment into model weights. The two axes that matter are
**who generates the tokens we train on** (off-policy vs on-policy) and **how much feedback each token gets**
(sparse vs dense).

| | Tokens trained on are sampled by | Feedback signal | Density | Main failure mode |
|---|---|---|---|---|
| **SFT** (supervised fine-tuning) | the teacher / dataset (**off-policy**) | cross-entropy on every target token | **dense** | exposure bias — the student never visits its own states during training |
| **RL / GRPO** | the student (**on-policy**) | one scalar reward per completion (here: Stockfish $Q(s,a)$), turned into a group-relative advantage | **sparse** | credit assignment: every token of a completion gets the same advantage |
| **On-policy distillation (OPD)** | the student (**on-policy**) | teacher log-probs on the *student's own prefix*, per token | **dense** | thinking-pattern mismatch / teacher has no new knowledge |

From the OPD notes (*"How to Learn On-Policy Distillation"*):

```
Traditional KD (off-policy):
  Teacher generates response → Student trains on fixed teacher tokens
  Problem: Student never visits its own states during training → exposure bias

OPD (on-policy):
  Student generates its own response → Teacher provides per-token feedback
  Advantage: Dense token-level reward signal on states the student actually visits
```

The OPD pipeline (student samples → teacher scores the **same** prefixes → per-token KL):

```text
Prompt x
   │
   ▼
Student πθ samples ŷ ──────────────────────────────┐
   │                                               │
   ▼                                               │
Student prefix ŷ<t ──► Teacher πT (same prefix)    │
   │                         │                     │
   ▼                         ▼                     │
pt (student)              qt (teacher)             │
   │                         │                     │
   └──────────► KL per token ◄─┘                   │
                   │                               │
                   ▼                               │
         Loss = Σ_t KL(pt ∥ qt)                    │
                   │                               │
                   ▼                               │
            Update θ ──────────────────────────────┘
```


OPD is "on-policy like RL, dense like SFT". §2–3 build the GRPO loss; §4 builds the OPD loss;
§5 puts the three signals next to each other on the same position.

## Why prompt will not work?

Lichess hands us **trillions of positions** and Stockfish is **close to an oracle** — so the raw signal exists. The difficulty is everything around it (Adithya Bhaskar's thread on the Queen paper, arXiv:2610.03695, makes the same points from a much bigger project):

1. **The data is not in the right format to learn explanations.** Engines emit centipawn evaluations and principal variations — not natural-language reasoning. There is no corpus of position → expert *explanation*; you have to manufacture one (Queen seeds theirs from a frontier model, then improves it).
2. **Even legality is hard for pretrained LMs.** Before GPT-5.6-Sol, no language model except Gemini could play legal moves consistently — the action space (~35 moves/ply, strict syntax) is adversarial to next-token prediction. Our own eval hits the same wall: legal-move rate is the *floor*, not the achievement.
3. **Explanations need search; LMs have none.** A good explanation states the best move, the predicted principal variation, *and why the alternatives are worse*. That is look-ahead — an LM's weights do not do it natively, it has to be distilled in (Queen's natural-language Bellman update; our GRPO + distillation stages).
4. **Playing strength is a prerequisite.** An explanation of a weak move is a confident wrong answer. This is why headline playing strength barely moved for us while behaviour changed — and why Queen needed 7 iterations (+915 Elo) before its prose was worth trusting.
5. **The expert's knowledge is latent, not verbal.** Stockfish (or a chess encoder) 'knows' the position but cannot speak; the LM can speak but does not know. Coupling the two is an architecture problem (cross-attention to a silent encoder in Queen) — a different failure surface than pure training.

So the recipe anyone converges on — labelled expert signal, group-relative RL from that signal, then distillation back into the policy — is not a fashion choice. It is the workaround for the five walls above. Sources: [arXiv:2610.03695](https://arxiv.org/abs/2610.03695) (Queen, Princeton PLI), [the thread](https://x.com/AdithyaNLP/status/2107123927525077505).


## 2. GRPO loss: toy-tensor version
![grpo loss](figures/grpo-loss.png)
GRPO in four pieces (from the GRPO-loss notes):

1. **Ratio** of token probabilities between the policy being updated and the policy that sampled
2. **Advantages** — group-relative, so no critic is needed
3. **Clipping** — so a single update can't move the ratio too far
4. **KL divergence** to a frozen reference model
Here $\pi_{ref}$ is the reference policy distribution and $\pi_\theta$ is the student policy


We use **G = 4** candidate moves for our position. Rewards are **real** (Stockfish); the per-token
log-probs are **hand-set tensors** so every number below can be recomputed by hand.

### Kullback–Leibler divergence
Alright, now let’s say we want to infer $p(z|X)$ using $q(z|X)$. The KL divergence then formulated as follows:
$$
D_{KL}[q[z|X)||p(z|X)] = \sum_z q(z|X) \log \frac{q(z|X)}{p(z|X)} = E_q[\log q(z|X) - \log p(z|X)]$$
### What KL-Divergence do?
* allows use to compare 2 probability distributions
* if $q$ is the same as  $p$, then KLD = 0
* if $q$ is different from  $p$, KLD > 0




### 2.1 Rewards from Stockfish

Four candidate moves for White: castling, the pin Bg5, a quiet pawn move, and the speculative
Bxf7+ sacrifice. Each gets the repo reward $r = 1 - \text{winprob}_{\text{after}}$ (§0).

![scoring 4](figures/scoring-four-chess-moves.gif)

In [5]:
TOY_MOVES = ["e1g1", "c1g5", "a2a3", "c4f7"]
toy_scored = [move_reward(FEN, m) for m in TOY_MOVES]
toy_rewards = torch.tensor([s["reward"] for s in toy_scored], dtype=torch.float64)
print(pd.DataFrame(toy_scored)[["move", "san", "cp_after", "reward"]].round(4).to_string(index=False))

move   san  cp_after  reward
e1g1   O-O        34  0.5488
c1g5   Bg5        38  0.5545
a2a3    a3        50  0.5715
c4f7 Bxf7+      -276  0.1696


Reading the rewards: the three quiet moves land within a few tens of centipawns of each other. At depth 12
that is inside engine noise, so don't over-read their order (it can even disagree with Stockfish's root
best move printed in §0). The sacrifice is the real outlier, and the advantages below are driven by it.

### 2.2 Group-relative advantage

GRPO replaces PPO's learned critic with the group statistics of the $G$ completions sampled for the
same prompt:

$$A_i = \frac{r_i - \operatorname{mean}(r)}{\operatorname{std}(r)}$$

We use the sample standard deviation (Bessel-corrected, `torch.std` default — what TRL does) and add a
tiny $\epsilon_{\text{std}} = 10^{-4}$ to the denominator. That handles the degenerate **std = 0** case: if all $G$
rewards are equal (e.g. all four moves illegal ⇒ all −1.0), every advantage is exactly 0 and the group
contributes **no policy gradient** — there is nothing to prefer.

In [6]:
STD_EPS = 1e-4


def group_advantages(rewards: torch.Tensor, eps: float = STD_EPS) -> torch.Tensor:
    # A_i = (r_i - mean(r)) / std(r); std = 0 -> all-zero advantages (via eps).
    if rewards.numel() < 2:
        return torch.zeros_like(rewards)
    return (rewards - rewards.mean()) / (rewards.std() + eps)


toy_adv = group_advantages(toy_rewards)
print("rewards   :", [round(x, 4) for x in toy_rewards.tolist()])
print(f"mean={toy_rewards.mean():.4f}  std={toy_rewards.std():.4f}")
print("advantages:", [round(x, 4) for x in toy_adv.tolist()])
print("sum of advantages (≈0 by construction):", round(toy_adv.sum().item(), 10))

# Degenerate group: every reward equal -> std = 0 -> zero advantages, no NaN.
print("std=0 case:", group_advantages(torch.tensor([-1.0, -1.0, -1.0, -1.0], dtype=torch.float64)).tolist())

rewards   : [0.5488, 0.5545, 0.5715, 0.1696]
mean=0.4611  std=0.1946
advantages: [0.4505, 0.4798, 0.5671, -1.4974]
sum of advantages (≈0 by construction): 0.0
std=0 case: [0.0, 0.0, 0.0, 0.0]


## 3. On-policy distillation

From the OPD notes, OPD minimises the sequence-level **reverse KL** over **student-generated** trajectories:

$$\mathcal{L}_{\text{OPD}}(\theta) = \mathbb{E}_{x \sim \mathcal{D}_x, \hat{y} \sim \pi_\theta} \left[ \sum_{t=1}^{T} D_{\text{KL}}(p_t \| q_t) \right]$$

where $p_t$ = student next-token distribution and $q_t$ = teacher next-token distribution, **both evaluated on
the student's own prefix** $\hat{y}_{<t}$.

Two of the three variants are computed here (the third, top-k, restricts the KL to the student's top-k tokens):

| Variant | Per-position quantity |
|---|---|
| **Sampled-token** (cheapest; unbiased Monte-Carlo estimator) | $\ell_t = \log p_t(\hat{y}_t) - \log q_t(\hat{y}_t)$ ||
| **Full-vocabulary** (densest, $O(BTM)$) | $D_{\text{KL}}(p_t \| q_t) = \sum_v p_t(v)\,[\log p_t(v) - \log q_t(v)]$ |

B is the batch size, how many sequences you score at once.
T is the sequence length, how many token positions are in each sequence.
M is the vocabulary size, every token the model could predict next. For this Qwen model that is about 151,000.

### The teacher

A real OPD run uses a **stronger teacher** (e.g. a larger Qwen3 or a thinking model). To stay CPU-feasible
**the teacher here is the same Qwen3-1.7B weights given privileged information**: its prompt additionally
contains Stockfish's best move and evaluation. It scores the **same student tokens** $\hat{y}$ with teacher
forcing. So $q_t$ = "what would the model predict next if it knew the engine's answer", and the per-token KL
shows *where the student's own trace departs from an engine-informed version of itself*.
This is context distillation, not a stronger-teacher result — the numbers illustrate the **mechanics**, not
the gains you'd get from a real teacher. (Per the notes, same-family teachers trained on the same data are
often "distributionally indistinguishable" — the privileged hint is what gives this teacher something new.)

The student samples at $T = 1$ with no top-k/top-p, so $\hat{y}_t \sim p_t$ exactly and $\ell_t$ is the unbiased
estimator the notes describe.

## 4. Why on-policy distillation helps exactly here

The game above shows the failure mode this talk keeps circling: **after one weak move the
model is in territory its training data never covered**. Off-policy SFT trains on Lichess
grandmaster positions; the base model's move 6 drags the game somewhere no grandmaster
game ever went, and from there SFT has *no gradient at all* — there is no labelled example
for the position the student actually created. This is exposure bias, and in chess it
compounds: one drifting move changes the whole character of the game, i.e. the *strategic*
part — precisely what the §6 table said is hardest to learn from text.

On-policy distillation attacks it directly, and the section above is already the worked
example:

- **Corrections in the states the student visits.** §4 sampled the *student's own*
  completion and had the teacher score those exact tokens. Scale that from one completion
  to whole self-play games and every weak move in the §6 record becomes a training
  signal — the teacher grades the student's game, not a grandmaster's.
- **Dense signal for a long-horizon problem.** GRPO (§2–§3) gives one scalar reward for a
  ~30-ply episode; the §5.1 figure made the contrast visible — the teacher's per-token
  log-probs supervise *every* generated token. For a game whose difficulty is exactly
  "one reward, thirty causes" (§6), that density is not a luxury, it is the only
  affordable gradient.
- **LUFFY-style mixing.** You do not have to choose between the two: LUFFY mixes
  off-policy expert moves into on-policy GRPO, so the sparse strategic signal from wins
  and the dense distillation signal coexist. This repo's planned M4 ablation — LUFFY
  on/off on **Qwen3-1.7B** (`docs/rewards.md`) — tests precisely whether that mixture
  beats GRPO-only on the horizon problem.

One sentence for the slide: *SFT teaches the student to imitate good games; on-policy
distillation teaches it to recognise how its own games go wrong.*

## 5. Fine Tuning Using LoRA (Low Rank Adaptation)
The idea: full fine-tuning updates one 4096×4096 weight matrix, which is about 16.8M trainable weights. LoRA freezes that matrix (W₀) and trains two thin matrices, B·A. At rank 8 that’s 65,536 weights, or 0.39%.
![idea](figures/1%20·%20The%20idea@2x.png)
Forward pass: a flow diagram with two paths. The input goes through the frozen W₀ on one path and through A → B → ×α/r on the other, and the two are added. The formula h = W₀x + (α/r)·BAx sits underneath.
![forward](figures/2%20·%20Forward%20pass@2x.png)
Training: four cards:
B starts at zero, so the model begins exactly as the base model.
Rank sets the capacity, with a bar chart of trainable weights per layer for ranks 4 to 64.
Memory savings come from only A and B needing gradients and optimizer state, with a note on QLoRA.
Placement covers which layers get adapters.
![Training](figures/3%20·%20Training@2x.png)
Deploying: you can merge the adapter back into the weights so the model runs at full speed, or keep one base model and swap small adapters per task. Each adapter is about 8 MB for a 7B model at rank 8 on the attention query and value layers.
![deploying](figures/4%20·%20Deploying%20adapters@2x.png)


## 6. Takeaways (slide mapping)

- **GRPO = "which of my own attempts was better?"** The group-relative advantage
  $A_i = (r_i - \operatorname{mean}(r))/\operatorname{std}(r)$ needs no critic, sums to zero within a group, and is
  exactly 0 when every sample scores the same — a group of identical (e.g. all-illegal) answers teaches nothing.
- **The ratio, clip and KL are guard-rails, not the signal.** The signal is the advantage; `ratio`/`clamp`
  bound how far one batch can push the policy (and correct sampler/trainer mismatch), and
  $e^{-\Delta} + \Delta - 1$ pulls toward $\pi_{\text{ref}}$ — often dropped ($\beta = 0$, DAPO-style) when the reward is a
  trustworthy verifier like Stockfish.
- **Aggregation is a design choice with behavioural consequences.** Sequence-mean (GRPO) down-weights tokens
  in long completions, so long wrong answers are penalised less per token; token-mean (DAPO) / constant
  normalisation (Dr. GRPO) remove that length bias.
- **Density: SFT and OPD are dense, GRPO is sparse — but only RL and OPD are on-policy.** In our example GRPO
  produced $G = 4$ numbers for the whole group, while OPD produced one per token on the student's own trace:
  OPD is "on-policy like RL, dense like SFT".
- **OPD only works if the teacher has something new *and* compatible to say.** Watch the overlap ratio and
  entropy gap before trusting it; for chess, Stockfish-as-reward (RL) and a strong chess-reasoning teacher
  (OPD) are complementary, not substitutes.

